# 4. Linear Regression

This notebook covers:
- **Simple linear regression** (one predictor)
- **Multiple linear regression** (several predictors)
- Regression **diagnostics**: residual plots, normality, multicollinearity
- Goodness of fit: $R^2$, adjusted $R^2$, RMSE
- Using both **scikit-learn** and **statsmodels**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
import statsmodels.api as sm
from scipy import stats

%matplotlib inline
np.random.seed(42)

## 4.1 Simple Linear Regression

Model: $y = \beta_0 + \beta_1 x + \varepsilon$

The OLS estimator minimizes $\sum_{i=1}^{n}(y_i - \hat{y}_i)^2$.

In [ ]:
# Generate data: study hours vs exam score
n = 50
hours = np.random.uniform(1, 10, n)
scores = 40 + 5.5 * hours + np.random.normal(0, 5, n)

# Fit with sklearn
model_sk = LinearRegression()
model_sk.fit(hours.reshape(-1, 1), scores)
print(f"Intercept: {model_sk.intercept_:.4f}")
print(f"Slope:     {model_sk.coef_[0]:.4f}")
print(f"R-squared: {model_sk.score(hours.reshape(-1, 1), scores):.4f}")

# Plot
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(hours, scores, alpha=0.6, edgecolors='k', s=40)
x_line = np.linspace(0, 11, 100)
ax.plot(x_line, model_sk.predict(x_line.reshape(-1, 1)), 'r-', lw=2, label='OLS fit')
ax.set_xlabel('Study Hours')
ax.set_ylabel('Exam Score')
ax.set_title('Simple Linear Regression')
ax.legend()
plt.tight_layout()
plt.show()

## 4.2 Statsmodels: Detailed Summary

Statsmodels provides a comprehensive regression summary with p-values, confidence intervals, and diagnostic statistics.

In [ ]:
X_sm = sm.add_constant(hours)  # Add intercept column
ols_model = sm.OLS(scores, X_sm).fit()
print(ols_model.summary())

## 4.3 Multiple Linear Regression

Model: $y = \beta_0 + \beta_1 x_1 + \beta_2 x_2 + \cdots + \beta_p x_p + \varepsilon$

**Adjusted $R^2$** penalizes for the number of predictors:
$$R^2_{\text{adj}} = 1 - \frac{(1 - R^2)(n-1)}{n - p - 1}$$

In [ ]:
# Simulated dataset: house price prediction
n = 200
area = np.random.uniform(50, 200, n)
rooms = np.random.randint(1, 6, n)
age = np.random.uniform(0, 50, n)
price = 20000 + 150 * area + 8000 * rooms - 500 * age + np.random.normal(0, 5000, n)

df = pd.DataFrame({'area': area, 'rooms': rooms, 'age': age, 'price': price})

X = df[['area', 'rooms', 'age']]
y = df['price']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model_mult = LinearRegression().fit(X_train, y_train)
y_pred = model_mult.predict(X_test)

print("Coefficients:")
for name, coef in zip(X.columns, model_mult.coef_):
    print(f"  {name:>6}: {coef:>10.2f}")
print(f"  intercept: {model_mult.intercept_:.2f}")
print(f"\nR-squared (test): {r2_score(y_test, y_pred):.4f}")
print(f"RMSE (test):      {np.sqrt(mean_squared_error(y_test, y_pred)):.2f}")

## 4.4 Regression Diagnostics

Key assumptions to check:
1. **Linearity**: residuals vs fitted should show no pattern
2. **Normality of residuals**: Q-Q plot should be approximately linear
3. **Homoscedasticity**: constant variance of residuals
4. **No multicollinearity**: check VIF (Variance Inflation Factor)

In [ ]:
# Fit full model with statsmodels for diagnostics
X_sm_full = sm.add_constant(X_train)
ols_full = sm.OLS(y_train, X_sm_full).fit()
residuals = ols_full.resid
fitted = ols_full.fittedvalues

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Residuals vs Fitted
axes[0].scatter(fitted, residuals, alpha=0.5, s=20)
axes[0].axhline(0, color='red', ls='--')
axes[0].set_xlabel('Fitted values')
axes[0].set_ylabel('Residuals')
axes[0].set_title('Residuals vs Fitted')

# Q-Q plot
stats.probplot(residuals, dist='norm', plot=axes[1])
axes[1].set_title('Q-Q Plot')

# Histogram of residuals
axes[2].hist(residuals, bins=20, edgecolor='black', alpha=0.7, color='steelblue', density=True)
x_norm = np.linspace(residuals.min(), residuals.max(), 100)
axes[2].plot(x_norm, stats.norm.pdf(x_norm, residuals.mean(), residuals.std()), 'r-', lw=2)
axes[2].set_title('Residual Distribution')

plt.tight_layout()
plt.show()

# VIF
from statsmodels.stats.outliers_influence import variance_inflation_factor
print("\nVariance Inflation Factors:")
for i, col in enumerate(X_train.columns):
    vif = variance_inflation_factor(X_sm_full.values, i + 1)
    print(f"  {col:>6}: {vif:.2f}")

## Key Takeaways

- **Simple regression** fits one predictor; **multiple regression** handles several
- Use **adjusted $R^2$** to compare models with different numbers of predictors
- Always check **residual diagnostics** before trusting regression results
- **VIF > 5** suggests problematic multicollinearity
- **statsmodels** provides inference (p-values, CIs); **sklearn** is better for prediction pipelines